In [1]:
!pip install -U datasets transformers evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 19.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
import os, shutil, glob
from collections import Counter
from typing import List
import numpy as np
import torch
from torch import nn
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline
)
from sklearn.metrics import classification_report

In [3]:
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


In [4]:
MODEL_NAME = 'ai-forever/ruRoberta-large'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

In [5]:
DATASET_NAME = 'Davlan/sib200'
DATASET_LANGUAGE = 'rus_Cyrl'

train_set = load_dataset(DATASET_NAME, DATASET_LANGUAGE, split='train')
validation_set = load_dataset(DATASET_NAME, DATASET_LANGUAGE, split='validation')
test_set = load_dataset(DATASET_NAME, DATASET_LANGUAGE, split='test')

README.md: 0.00B [00:00, ?B/s]

train.tsv: 0.00B [00:00, ?B/s]

dev.tsv: 0.00B [00:00, ?B/s]

test.tsv: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [6]:
MINIBATCH_SIZE = 4

tokenized_train_set = train_set.map(
    lambda it: tokenizer(it['text'], truncation=True, max_length=512),
    batched=True,
    batch_size=MINIBATCH_SIZE
)

tokenized_validation_set = validation_set.map(
    lambda it: tokenizer(it['text'], truncation=True, max_length=512),
    batched=True,
    batch_size=MINIBATCH_SIZE
)

Map:   0%|          | 0/701 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

In [7]:
list_of_categories = sorted(list(
    set(train_set['category']) |
    set(validation_set['category']) |
    set(test_set['category'])
))

indices_of_categories = list(range(len(list_of_categories)))
n_categories = len(list_of_categories)

id2label = dict(zip(indices_of_categories, list_of_categories))
label2id = dict(zip(list_of_categories, indices_of_categories))

In [8]:
labeled_train_set = tokenized_train_set.add_column(
    'label',
    [label2id[val] for val in tokenized_train_set['category']]
)

labeled_validation_set = tokenized_validation_set.add_column(
    'label',
    [label2id[val] for val in tokenized_validation_set['category']]
)

In [9]:
from collections import Counter
from torch import nn

label_counts = Counter(labeled_train_set['label'])

weights_list = [0] * n_categories
total = sum(label_counts.values())
for label_id, count in label_counts.items():
    weights_list[label_id] = total / (n_categories * count)

class_weights = torch.tensor(weights_list, dtype=torch.float).cuda()

In [10]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [11]:
classifier = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=n_categories,
    id2label=id2label,
    label2id=label2id
).cuda()

for param in classifier.parameters():
    param.data = param.data.contiguous()

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at ai-forever/ruRoberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
encoder = classifier.roberta.encoder
for layer in encoder.layer[-5:]:
    for module in layer.modules():
        classifier._init_weights(module)

In [13]:
cls_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    res = cls_metric.compute(predictions=predictions, references=labels, average='macro')
    return {'f1': res['f1']}

In [14]:
training_args = TrainingArguments(
    output_dir='ruberta_sib200_stable',
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    num_train_epochs=20,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    warmup_ratio=0.2,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=1,
    report_to='none',
    fp16=False,
)

In [ ]:
trainer = WeightedTrainer(
    model=classifier,
    args=training_args,
    train_dataset=labeled_train_set,
    eval_dataset=labeled_validation_set,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

/tmp/ipykernel_55/3249069836.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedTrainer(


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Epoch,Training Loss,Validation Loss,F1
1,No log,1.895838,0.082890
2,No log,0.662810,0.822730
3,No log,0.489775,0.832520
4,No log,0.613118,0.855227
5,No log,0.662760,0.891188
6,No log,0.688910,0.873622
7,No log,0.827882,0.872712
8,No log,0.784352,0.894718
9,No log,0.830219,0.884522
10,No log,0.825689,0.894718


TrainOutput(global_step=880, training_loss=0.21133962011134083, metrics={'train_runtime': 527.1155, 'train_samples_per_second': 26.598, 'train_steps_per_second': 1.669, 'total_flos': 1121434524841788.0, 'train_loss': 0.21133962011134083, 'epoch': 20.0})

In [ ]:
classification_pipeline = pipeline(
    'text-classification',
    model=classifier,
    tokenizer=tokenizer,
    device=0
)

texts = list(test_set['text'])

raw_predictions = classification_pipeline(texts, batch_size=16, truncation=True, max_length=512)

y_pred = [pred['label'] for pred in raw_predictions]
y_true = test_set['category']

print(classification_report(y_true=y_true, y_pred=y_pred))

Device set to use cuda:0


                    precision    recall  f1-score   support

     entertainment       0.86      0.63      0.73        19
         geography       0.94      0.88      0.91        17
            health       0.92      1.00      0.96        22
          politics       0.97      1.00      0.98        30
science/technology       0.88      0.96      0.92        51
            sports       0.96      0.88      0.92        25
            travel       0.90      0.90      0.90        40

          accuracy                           0.91       204
         macro avg       0.92      0.89      0.90       204
      weighted avg       0.91      0.91      0.91       204

